# 04b — Social Media Charts (Pillow)
Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to the `web` preset (1664×936px) with watermark.

**Rendering engine:** Pillow (PIL) — pixel-level control, no browser dependency.

**Production Charts (posting order):**
1. Top 10 Causes: Female vs Male (side-by-side, per 100k)
2. Top 10 Causes: White vs Black (side-by-side, per 100k)
3. National Abortion Comparison (stacked male/female with gestation breakdown)
3b. Top 5 Causes: White Americans (stacked + abortion) — supplemental
3c. Top 5 Causes: Black Americans (stacked + abortion) — supplemental
4. Per-Capita: White vs Black side-by-side (rate per 100k, shared scale)

**Data source:** All chart data stored in DuckDB `chart_*` tables.
Edit data in `04-viz.ipynb`, then re-run the table creation script if needed.


In [ ]:
# ===================================================================
# SETUP
# ===================================================================

import sys
import os
from pathlib import Path

import duckdb
import yaml

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT / 'src'))
sys.path.insert(0, str(PROJECT.parent.parent / 'shared'))

from chart_factory import render_chart as _render_chart_factory
import copy as _copy

# Capture every rendered chart's config so we can produce a matching WEB
# version (title/subtitle/source stripped, larger data) without restating any
# config — the web charts are derived from the exact same specs as the social
# ones, so they can never drift. See the 'WEB VERSIONS' cell at the end.
_CHART_CONFIGS = []

def render_chart(config):
    # Record a copy for the web pass (drop the live db handle; re-injected later).
    _cap = {k: v for k, v in config.items() if k != 'db'}
    _CHART_CONFIGS.append(_copy.deepcopy(_cap))
    return _render_chart_factory(config)

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Create outputs/social + outputs/web directories
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)
web_dir = PROJECT / 'outputs' / 'web'
web_dir.mkdir(parents=True, exist_ok=True)

# Load annotation metadata
annotations = conn.execute("""
  SELECT key, value FROM chart_annotations
""").df().set_index('key')['value'].to_dict()

print('\u2713 chart_factory loaded')
print(f'\u2713 DuckDB connected')
print(f'\u2713 Charts export to: {social_dir}')
print(f'\u2713 Annotations: {annotations}')

In [ ]:
# ===================================================================
# COLOR PALETTE — edit here to experiment, re-run to update all charts
# ===================================================================

# --- Sex chart colors (Charts 1, 2, 3, 4) ---
C_MALE = '#0A9396'          # Dark Cyan
C_FEMALE = '#EE9B00'        # Golden Orange
C_ABORTION = '#AE2012'      # Oxidized Iron (accent)
C_SUICIDE_HL = '#94D2BD'    # Suicide highlight on male panel

# --- Race chart colors (Chart 1b) ---
C_WHITE = '#023047'         # Deep Space Blue
C_BLACK = '#6C757D'         # Slate Grey
C_HISPANIC = '#E9D8A6'      # Vanilla Cream (warm accent)
C_RACE_SUICIDE_HL = '#0A9396'   # Suicide highlight on white panel
C_RACE_HOMICIDE_HL = '#EE9B00'  # Homicide highlight on black panel

# --- Detail bar gradients (4 steps: lightest to darkest) ---
GRAD_SUICIDE_MALE = ['#4FB3AA', '#94D2BD','#BFD5B2','#E9D8A6']
GRAD_SUICIDE_RACE = ['#4FB3AA', '#94D2BD','#BFD5B2','#E9D8A6']
GRAD_HOMICIDE_RACE = ['#EE9B00','#DC8101', '#D37402', '#CA6702']

# --- In-bar text color ---
C_BAR_TEXT = '#023047'

print('\u2713 Palette loaded')
print(f'  Male: {C_MALE}  |  Female: {C_FEMALE}  |  Abortion: {C_ABORTION}')
print(f'  White: {C_WHITE}  |  Black: {C_BLACK}  |  Hispanic: {C_HISPANIC}')

---

## Chart 1: Top 10 Causes of Death — Female vs Male (Side-by-Side)

Two horizontal bar panels: Female top 10 on the left, Male top 10 on the right.
Each panel sorted independently by rate per 100k. Suicide highlighted on male panel
with annotation showing its rank for women.

In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_female_top10',
    'table_right': 'chart_male_top10',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Top 10 Causes of Death: Female vs Male',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'Female',
    'right_title': 'Male',
    'source': 'CDC WONDER 2024',

    # Colors
    'left_color': C_FEMALE,
    'right_color': C_MALE,

    # Highlight
    'highlight_right': {
        'category': 'Suicide',
        'color': C_SUICIDE_HL,
        'annotation': f'#{annotations["suicide_female_rank"]} for women',
    },

    # Detail bar
    'detail_bar': {
        'table': 'chart_male_suicide_age',
        'title': 'Male suicide by age:',
        'gradient': GRAD_SUICIDE_MALE,
    },

    # Export
    'filename': '01_top10_causes_female_vs_male',
})

---
## Chart 2: Top 10 Causes of Death — White vs Black (Side-by-Side)
Same pattern but comparing races. Suicide highlighted on white panel,
homicide highlighted on black panel. Two detail bars: white suicide by age,
black homicide victims by offender race.


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_white_top10',
    'table_right': 'chart_black_top10',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Top 10 Causes of Death: White vs Black',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'White',
    'right_title': 'Black',
    'source': 'CDC WONDER 2024 · FBI SHR 2024',

    # Colors
    'left_color': C_WHITE,
    'right_color': C_BLACK,

    # Highlights
    'highlight_left': {
        'category': 'Suicide',
        'color': C_RACE_SUICIDE_HL,
        'annotation': f'#{annotations["suicide_black_rank"]} for Black',
    },
    'highlight_right': {
        'category': 'Homicide',
        'color': C_RACE_HOMICIDE_HL,
        'annotation': f'#{annotations["homicide_white_rank"]} for White',
    },

    # Detail bars
    'detail_bars': [
        {
            'table': 'chart_white_suicide_age',
            'title': 'White suicide by age:',
            'gradient': GRAD_SUICIDE_RACE,
        },
        {
            'table': 'chart_black_homicide_offender',
            'title': 'Black homicide victims \u2014 offender race:',
            'gradient': GRAD_HOMICIDE_RACE,
        },
    ],

    # Export
    'filename': '02_top10_causes_white_vs_black',
})

---
## Chart 2b: Top 10 Causes of Death — White vs Hispanic (Side-by-Side)


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_white_top10',
    'table_right': 'chart_hispanic_top10',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Top 10 Causes of Death: White vs Hispanic',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'White',
    'right_title': 'Hispanic',
    'source': 'CDC WONDER 2024',

    # Colors
    'left_color': C_WHITE,
    'right_color': C_HISPANIC,

    # Export
    'filename': '02b_top10_causes_white_vs_hispanic',
})


---
## Chart 2c: Top 10 Causes of Death — Black vs Hispanic (Side-by-Side)


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_black_top10',
    'table_right': 'chart_hispanic_top10',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Top 10 Causes of Death: Black vs Hispanic',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'Black',
    'right_title': 'Hispanic',
    'source': 'CDC WONDER 2024',

    # Colors
    'left_color': C_BLACK,
    'right_color': C_HISPANIC,

    # Export
    'filename': '02c_top10_causes_black_vs_hispanic',
})


---
## Chart 2d: Top 10 Causes — White vs Black vs Hispanic (3-Way)

Experimental — likely too cramped for mobile viewing.


In [ ]:
# 3-way comparison: render as three single_ranked_bars panels side by side
# Using Pillow directly since chart_factory doesn't have a 3-panel template
from PIL import Image as PILImage
from chart_templates import single_ranked_bars, _draw_title_block, _draw_footer, _get_font
from chart_templates import _set_render_scale, _reset_render_scale, _s
from viz import PRESETS
import duckdb as _ddb

img_w, img_h, _ = PRESETS['twitter_landscape']
# This cell hand-builds a 3-panel chart (no factory template exists) using the
# scale-aware helpers, so set the render scale to match the canvas.
_set_render_scale(img_w)
img = PILImage.new('RGB', (img_w, img_h), (255, 255, 255))
from PIL import ImageDraw
draw = ImageDraw.Draw(img)

# Title
from chart_templates import _draw_title_block, _draw_footer, MARGIN_LR
content_top = _draw_title_block(draw, 'Top 10 Causes of Death by Race/Ethnicity',
                                 'Rate per 100,000 population')

# Load data for 3 panels
df_w = conn.execute('SELECT * FROM chart_white_top10').df()
df_b = conn.execute('SELECT * FROM chart_black_top10').df()
df_h = conn.execute('SELECT * FROM chart_hispanic_top10').df()

panel_w = (img_w - _s(MARGIN_LR) * 2 - _s(40)) // 3  # 3 panels with gaps
panel_h = img_h - content_top - _s(60)
colors = [C_WHITE, C_BLACK, C_HISPANIC]
titles = ['White', 'Black', 'Hispanic']
dfs = [df_w, df_b, df_h]

# Render each panel as a mini ranked bar chart
from chart_templates import _hex_to_rgb, _draw_rounded_rect, _text_width, _text_y_centered
font_label = _get_font(10)
font_value = _get_font(10, bold=True)
font_panel_title = _get_font(14, bold=True)

for panel_idx, (df_panel, color, ptitle) in enumerate(zip(dfs, colors, titles)):
    x_offset = _s(MARGIN_LR) + panel_idx * (panel_w + _s(20))
    y_start = content_top + _s(5)
    
    # Panel title
    draw.text((x_offset, y_start), ptitle, fill=_hex_to_rgb(color), font=font_panel_title)
    y_start += _s(24)
    
    bar_h = max((panel_h - _s(30)) // 10 - _s(4), _s(12))
    max_val = df_panel['rate_per_100k'].max()
    bar_area = panel_w - _s(120)  # space for label + bar + value
    
    for row_idx, (_, row) in enumerate(df_panel.iterrows()):
        y = y_start + row_idx * (bar_h + _s(4))
        cause = row['cause']
        val = row['rate_per_100k']
        
        # Truncate long labels
        label = cause[:12] + '..' if len(cause) > 14 else cause
        draw.text((x_offset, y + _s(1)), label, fill=(55, 65, 81), font=font_label)
        
        # Bar
        bar_x = x_offset + _s(90)
        bar_w = int((val / max_val) * (bar_area - _s(40)))
        _draw_rounded_rect(draw, bar_x, y, bar_x + max(bar_w, _s(3)), y + bar_h,
                           fill=_hex_to_rgb(color))
        
        # Value
        draw.text((bar_x + bar_w + _s(4), y + _s(1)), f'{val:.0f}',
                  fill=(55, 65, 81), font=font_value)

_draw_footer(draw, img_w, img_h, source='CDC WONDER 2024')
_reset_render_scale()

# Save
out_path = social_dir / '02d_top10_causes_3way.png'
img.save(out_path, format='PNG', optimize=True)
print(f'Saved -> {out_path} ({img_w}x{img_h})')

from IPython.display import Image, display
display(Image(filename=str(out_path)))


---
## Chart 3: National Abortion Comparison
What if abortion were counted as a cause of death?
Stacked male/female bars for top 5 causes, with abortion as a solid accent bar.
Inner segments show gestation timing.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_national_stacked',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Inner segments: gestation age dividers inside the Abortion bar
    'inner_segments': {
        'category': 'Abortion',
        'table': 'chart_abortion_gestation',
        'text_color': '#E9D8A6',
        'divider_color': '#fff5e6',
        'min_width_pct': 10.0,
    },

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes of death, by sex',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024 | Gestational age estimate based on CDC Abortion Surveillance 2022',

    # Export
    'filename': '03_abortion_comparison_national',
})

---
## Chart 3b: Top 5 Causes — White Americans (+ Abortion)
Same stacked format filtered to White Americans.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_stacked_white',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes among White Americans, by sex',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',

    # Export
    'filename': '03b_top5_causes_white',
})

---
## Chart 3c: Top 5 Causes — Black Americans (+ Abortion)
Same stacked format filtered to Black Americans.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_stacked_black',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes among Black Americans, by sex',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',

    # Export
    'filename': '03c_top5_causes_black',
})

---
## Chart 3d: Top 5 Causes — Hispanic Americans (+ Abortion)
Same stacked format for Hispanic/Latino population.


In [ ]:
render_chart({
    'type': 'stacked_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table': 'chart_stacked_hispanic',
    'category_col': 'cause',

    # Segments
    'segments': [
        {'value_col': 'male_deaths', 'label': 'Male', 'color': C_MALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'female_deaths', 'label': 'Female', 'color': C_FEMALE, 'text_color': C_BAR_TEXT},
        {'value_col': 'abortion_deaths', 'label': 'Abortion', 'color': C_ABORTION, 'text_color': '#E9D8A6'},
    ],

    # Text
    'title': 'What if abortion were counted as a cause of death?',
    'subtitle': 'Top 5 causes among Hispanic Americans, by sex',
    'source': 'CDC WONDER 2024 \u00b7 Guttmacher Institute 2024',

    # Export
    'filename': '03d_top5_causes_hispanic',
})


---
## Chart 4: Per-Capita Comparison — White vs Black
Same causes, expressed as a rate per 100,000 population.
Denominator includes aborted (if not aborted, they would be counted as population).
This allows direct cross-race comparison on a shared scale.


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',
    # Data
    'table_left': 'chart_white_percapita',
    'table_right': 'chart_black_percapita',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',
    # Text
    'title': 'Abortion as a cause of death: White vs Black',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'White',
    'right_title': 'Black',
    'source': 'CDC WONDER 2024 · Guttmacher Institute 2024',
    # Colors
    'left_color': C_WHITE,
    'right_color': C_BLACK,
    # Highlight abortion bars
    'highlight_left': {'category': 'Abortion', 'color': C_ABORTION},
    'highlight_right': {'category': 'Abortion', 'color': C_ABORTION},
    # Export
    'filename': '04_percapita_white_vs_black',
})


---
## Chart 4b: Per-Capita Comparison — White vs Hispanic


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_white_percapita',
    'table_right': 'chart_hispanic_percapita',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Abortion as a cause of death: White vs Hispanic',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'White',
    'right_title': 'Hispanic',
    'source': 'CDC WONDER 2024 \u00b7 Guttmacher Institute 2024',

    # Colors
    'left_color': C_WHITE,
    'right_color': C_HISPANIC,

    # Highlight abortion bars
    'highlight_left': {'category': 'Abortion', 'color': C_ABORTION},
    'highlight_right': {'category': 'Abortion', 'color': C_ABORTION},

    # Export
    'filename': '04b_percapita_white_vs_hispanic',
})


---
## Chart 4c: Per-Capita Comparison — Black vs Hispanic


In [ ]:
render_chart({
    'type': 'side_by_side_bars',
    'db': conn,
    'cfg': cfg,
    'preset': 'twitter_landscape',

    # Data
    'table_left': 'chart_black_percapita',
    'table_right': 'chart_hispanic_percapita',
    'category_col': 'cause',
    'value_col': 'rate_per_100k',
    'label_col': 'total_label',

    # Text
    'title': 'Abortion as a cause of death: Black vs Hispanic',
    'subtitle': 'Rate per 100,000 population',
    'left_title': 'Black',
    'right_title': 'Hispanic',
    'source': 'CDC WONDER 2024 \u00b7 Guttmacher Institute 2024',

    # Colors
    'left_color': C_BLACK,
    'right_color': C_HISPANIC,

    # Highlight abortion bars
    'highlight_left': {'category': 'Abortion', 'color': C_ABORTION},
    'highlight_right': {'category': 'Abortion', 'color': C_ABORTION},

    # Export
    'filename': '04c_percapita_black_vs_hispanic',
})


---
## Web versions (for the project page)

Re-render every chart above in **web mode**: the title, subtitle, and source
footer are dropped (the page's markdown already carries that framing), which
hands that vertical space to the data so the chart reads without zooming. Only
the `@unwelcomedata` watermark is kept, so a downloaded copy still carries
attribution. Web charts go to `outputs/web/` (social charts in `outputs/social/`
are untouched). These are derived from the exact same configs as the social
charts, so they can never drift out of sync.

In [ ]:
# Point the factory's export dir at outputs/web for this pass, then restore.
_orig_out_social = cfg['paths'].get('outputs_social')
cfg['paths']['outputs_social'] = str(web_dir)

for _spec in _CHART_CONFIGS:
    _c = _copy.deepcopy(_spec)
    _c['db'] = conn                 # re-inject the live connection
    _c['cfg'] = cfg
    _c['preset'] = 'web'            # page-sized canvas
    _c['web_mode'] = True           # strip title/subtitle/source, keep watermark
    _c['display'] = False           # don't double-display inline
    _render_chart_factory(_c)

cfg['paths']['outputs_social'] = _orig_out_social  # restore
print(f'\u2713 Web charts written to {web_dir}')

In [ ]:
# Web version of the hand-built 3-way chart (Chart 2d). It doesn't go through
# render_chart, so it isn't in _CHART_CONFIGS — build its web variant directly.
# Web mode: no title block, watermark-only footer, taller panels.
from chart_templates import (_set_render_scale, _reset_render_scale, _s,
                             _draw_title_block, _draw_footer, _get_font,
                             _hex_to_rgb, _draw_rounded_rect, MARGIN_LR)
from viz import PRESETS

_img_w, _img_h, _ = PRESETS['web']
_set_render_scale(_img_w)
_wimg = PILImage.new('RGB', (_img_w, _img_h), (255, 255, 255))
_wdraw = ImageDraw.Draw(_wimg)

# web_mode title block => small top pad only, no title/subtitle.
_content_top = _draw_title_block(_wdraw, '', None, web_mode=True)

_dfw = conn.execute('SELECT * FROM chart_white_top10').df()
_dfb = conn.execute('SELECT * FROM chart_black_top10').df()
_dfh = conn.execute('SELECT * FROM chart_hispanic_top10').df()

_panel_w = (_img_w - _s(MARGIN_LR) * 2 - _s(40)) // 3
_panel_h = _img_h - _content_top - _s(50)
_colors = [C_WHITE, C_BLACK, C_HISPANIC]
_titles = ['White', 'Black', 'Hispanic']
_dfs = [_dfw, _dfb, _dfh]

_font_label = _get_font(10)
_font_value = _get_font(10, bold=True)
_font_panel_title = _get_font(14, bold=True)

for _pi, (_dfp, _color, _pt) in enumerate(zip(_dfs, _colors, _titles)):
    _xoff = _s(MARGIN_LR) + _pi * (_panel_w + _s(20))
    _ys = _content_top + _s(5)
    _wdraw.text((_xoff, _ys), _pt, fill=_hex_to_rgb(_color), font=_font_panel_title)
    _ys += _s(24)
    _bar_h = max((_panel_h - _s(30)) // 10 - _s(4), _s(12))
    _maxv = _dfp['rate_per_100k'].max()
    _bar_area = _panel_w - _s(120)
    for _ri, (_, _row) in enumerate(_dfp.iterrows()):
        _y = _ys + _ri * (_bar_h + _s(4))
        _cause = _row['cause']; _val = _row['rate_per_100k']
        _label = _cause[:12] + '..' if len(_cause) > 14 else _cause
        _wdraw.text((_xoff, _y + _s(1)), _label, fill=(55, 65, 81), font=_font_label)
        _bx = _xoff + _s(90)
        _bw = int((_val / _maxv) * (_bar_area - _s(40)))
        _draw_rounded_rect(_wdraw, _bx, _y, _bx + max(_bw, _s(3)), _y + _bar_h, fill=_hex_to_rgb(_color))
        _wdraw.text((_bx + _bw + _s(4), _y + _s(1)), f'{_val:.0f}', fill=(55, 65, 81), font=_font_value)

# Watermark-only footer.
_draw_footer(_wdraw, _img_w, _img_h, source=None, web_mode=True)
_reset_render_scale()

_wimg.save(web_dir / '02d_top10_causes_3way.png', format='PNG', optimize=True)
print(f'\u2713 Web 3-way chart written to {web_dir}/02d_top10_causes_3way.png')

---

## Summary & Cleanup

In [ ]:
conn.close()

# Verify all exports
pngs = sorted(social_dir.glob('*.png'))
web_pngs = sorted(web_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n\u2713 Social charts ({len(pngs)}) in {social_dir} (twitter_landscape 1600\u00d7900, full title/source/watermark):')
for png in pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  \u2022 {png.name} ({size_kb:.0f} KB)')
print(f'\n\u2713 Web charts ({len(web_pngs)}) in {web_dir} (web preset 1664\u00d7936, no title/source, watermark only):')
for png in web_pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  \u2022 {png.name} ({size_kb:.0f} KB)')

print('\nSocial charts \u2192 Bluesky/X. Web charts \u2192 the project page (docs/).')